# 4. Exploratory Data Analysis with SQL (SQLite)

In [5]:
from pathlib import Path
import requests
import json
import pandas as pd

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

urls = [
    "https://gateway.pipeworx.io/spacex/v4/launches",
    "https://api.spacexdata.com/v4/launches",
]

launches = None

for url in urls:
    try:
        print("Trying:", url)
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        if isinstance(data, list) and len(data) > 0:
            launches = data
            print("Successfully collected", len(launches), "launch records")
            break

        print("Unexpected response format:", type(data).__name__)

    except Exception as e:
        print("Request failed:", str(e))

if launches is None:
    raise RuntimeError(
        "Could not retrieve launch data. Check the errors above."
    )

json_path = DATA_DIR / "spacex_launches.json"
csv_path = DATA_DIR / "spacex_launches.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(launches, f, indent=2)

pd.json_normalize(launches).to_csv(csv_path, index=False)

print("\nSaved files:")
print(json_path)
print(csv_path)
print("CSV shape:", pd.read_csv(csv_path).shape)
print("Columns:", pd.json_normalize(launches).columns.tolist())

Trying: https://gateway.pipeworx.io/spacex/v4/launches
Successfully collected 101 launch records

Saved files:
/content/data/spacex_launches.json
/content/data/spacex_launches.csv
CSV shape: (101, 10)
Columns: ['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia']


In [7]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

# Paths
DATA_DIR = Path("/content/data")
json_path = DATA_DIR / "spacex_launches.json"
cleaned_path = DATA_DIR / "spacex_launches_cleaned.csv"
db_path = DATA_DIR / "spacex_capstone.sqlite"

# Load collected data
with open(json_path, "r", encoding="utf-8") as f:
    launches = json.load(f)

df = pd.json_normalize(launches)

# Parse launch dates and extract year
df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce", utc=True)
df["year"] = df["date_utc"].dt.year

# Remove duplicate records, if any
df = df.drop_duplicates()

# Save cleaned CSV
df.to_csv(cleaned_path, index=False)

# Create SQLite database
with sqlite3.connect(db_path) as conn:
    df.to_sql("launches", conn, if_exists="replace", index=False)

    row_count = pd.read_sql_query(
        "SELECT COUNT(*) AS row_count FROM launches", conn
    ).iloc[0, 0]

print("Cleaning and database creation completed!")
print("Launch records:", len(df))
print("Database rows:", row_count)
print("Cleaned CSV:", cleaned_path)
print("SQLite database:", db_path)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 records:")
display(df.head())

Cleaning and database creation completed!
Launch records: 101
Database rows: 101
Cleaned CSV: /content/data/spacex_launches_cleaned.csv
SQLite database: /content/data/spacex_capstone.sqlite

Columns: ['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia', 'year']

First 5 records:


,name,status,success,date_utc,pad,rocket,details,links.article,links.webcast,links.wikipedia,year
0,Falcon Heavy | NROL-97,Launch Successful,True,2026-10-02 03:54:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
1,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,Launch Successful,True,2026-10-01 18:32:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
2,Falcon 9 Block 5 | Crew-13,Launch Successful,True,2026-10-01 15:10:06+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
3,Starship | Starlink Group 31-1 (Starship Fligh...,Launch Successful,True,2026-09-28 12:48:59+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026
4,Falcon 9 Block 5 | USSF-385,Launch Successful,True,2026-09-26 14:00:54+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2026


In [8]:
import sqlite3
import pandas as pd

db_path = "/content/data/spacex_capstone.sqlite"

with sqlite3.connect(db_path) as conn:
    print(pd.read_sql_query("SELECT * FROM launches LIMIT 5", conn))
    print("\nLaunches per year:")
    print(pd.read_sql_query("""
        SELECT year, COUNT(*) AS total_launches
        FROM launches
        WHERE year IS NOT NULL
        GROUP BY year
        ORDER BY year
    """, conn))

                                                name             status  \
0                             Falcon Heavy | NROL-97  Launch Successful   
1  Falcon 9 Block 5 | Transporter 18 (Dedicated S...  Launch Successful   
2                         Falcon 9 Block 5 | Crew-13  Launch Successful   
3  Starship | Starlink Group 31-1 (Starship Fligh...  Launch Successful   
4                        Falcon 9 Block 5 | USSF-385  Launch Successful   

   success                   date_utc   pad rocket details links.article  \
0        1  2026-10-02 03:54:00+00:00  None   None    None          None   
1        1  2026-10-01 18:32:00+00:00  None   None    None          None   
2        1  2026-10-01 15:10:06+00:00  None   None    None          None   
3        1  2026-09-28 12:48:59+00:00  None   None    None          None   
4        1  2026-09-26 14:00:54+00:00  None   None    None          None   

  links.webcast links.wikipedia  year  
0          None            None  2026  
1          N

In [10]:
import sqlite3
import pandas as pd
from IPython.display import display

db_path = "/content/data/spacex_capstone.sqlite"

with sqlite3.connect(db_path) as conn:
    # Inspect the actual database schema
    columns = pd.read_sql_query(
        "PRAGMA table_info(launches)", conn
    )["name"].tolist()

    print("Available columns:", columns)

    queries = {
        "Launches by site": """
            SELECT pad AS launch_site,
                   COUNT(*) AS launch_count
            FROM launches
            GROUP BY pad
            ORDER BY launch_count DESC;
        """,

        "Yearly launch counts": """
            SELECT year,
                   COUNT(*) AS launch_count
            FROM launches
            WHERE year IS NOT NULL
            GROUP BY year
            ORDER BY year;
        """,

        "Success rate by site": """
            SELECT pad AS launch_site,
                   COUNT(success) AS known_outcomes,
                   ROUND(
                       100.0 * SUM(
                           CASE WHEN success = 1 THEN 1 ELSE 0 END
                       ) / NULLIF(COUNT(success), 0), 2
                   ) AS success_rate_pct
            FROM launches
            GROUP BY pad
            ORDER BY success_rate_pct DESC;
        """,

        "Launches by rocket": """
            SELECT rocket AS rocket_id,
                   COUNT(*) AS launch_count,
                   SUM(
                       CASE WHEN success = 1 THEN 1 ELSE 0 END
                   ) AS successful_launches,
                   COUNT(success) AS known_outcomes
            FROM launches
            GROUP BY rocket
            ORDER BY launch_count DESC;
        """,

        "Launch records": """
            SELECT name, date_utc, pad, rocket, success, status
            FROM launches
            ORDER BY date_utc;
        """
    }

    for title, sql in queries.items():
        print(f"\n### {title}")
        display(pd.read_sql_query(sql, conn))

    # Payload analysis requires payload_mass_kg, which may not exist.
    if "payload_mass_kg" in columns:
        print("\n### Payload mass by outcome")
        display(pd.read_sql_query("""
            SELECT success,
                   COUNT(*) AS launches,
                   ROUND(AVG(payload_mass_kg), 2)
                       AS avg_payload_mass_kg
            FROM launches
            GROUP BY success;
        """, conn))
    else:
        print(
            "\nPayload mass analysis skipped: "
            "payload_mass_kg is not available in this dataset."
        )

Available columns: ['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia', 'year']

### Launches by site


,launch_site,launch_count
0,None,100
1,Space Launch Complex 4E,1



### Yearly launch counts


,year,launch_count
0,2026,101



### Success rate by site


...


### Launches by rocket


,rocket_id,launch_count,successful_launches,known_outcomes
0,None,100,100,100
1,Falcon 9 Block 5,1,0,0



### Launch records


,name,date_utc,pad,rocket,success,status
0,Falcon 9 Block 5 | Crew-12,2026-02-13 10:15:56+00:00,None,None,1.0,Launch Successful
1,Falcon 9 Block 5 | Starlink Group 17-13,2026-02-15 01:59:59+00:00,None,None,1.0,Launch Successful
2,Falcon 9 Block 5 | Starlink Group 6-103,2026-02-16 07:59:40+00:00,None,None,1.0,Launch Successful
3,Falcon 9 Block 5 | Starlink Group 10-36,2026-02-20 01:41:40+00:00,None,None,1.0,Launch Successful
4,Falcon 9 Block 5 | Starlink Group 17-25,2026-02-21 09:04:19+00:00,None,None,1.0,Launch Successful
...,...,...,...,...,...,...
96,Starship | Starlink Group 31-1 (Starship Fligh...,2026-09-28 12:48:59+00:00,None,None,1.0,Launch Successful
97,Falcon 9 Block 5 | Crew-13,2026-10-01 15:10:06+00:00,None,None,1.0,Launch Successful
98,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,2026-10-01 18:32:00+00:00,None,None,1.0,Launch Successful
99,Falcon Heavy | NROL-97,2026-10-02 03:54:00+00:00,None,None,1.0,Launch Successful



Payload mass analysis skipped: payload_mass_kg is not available in this dataset.


In [11]:
# Time analysis: launches by UTC hour (where timestamp parsing is valid)
df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce", utc=True)
df["utc_hour"] = df["date_utc"].dt.hour
df.to_sql("launches_with_hour", conn, if_exists="replace", index=False)
display(pd.read_sql_query('''
SELECT utc_hour, COUNT(*) AS launch_count
FROM launches_with_hour
WHERE utc_hour IS NOT NULL
GROUP BY utc_hour ORDER BY launch_count DESC
''', conn))

,utc_hour,launch_count
0,14,10
1,15,9
2,10,8
3,7,7
4,3,7
5,2,7
6,1,7
7,11,6
8,9,6
9,12,5


Interpret each SQL result using the actual table. Success-rate denominators exclude unknown outcomes. Save relevant query outputs or screenshots for the final presentation.